# Multilayer perceptron (MLP)

## n-gram model
We built a [character-level bigram](https://n03an.me/gpt/01_Bigram.html) model trained on a dataset of 32,000 names. Lets build a n-gram model using similar appraoch where n=4 i.e. 4-gram model.

### Shape of Input, Output and Weights
instead of single-character context, we will use 3-character sequence as input context and the next character as output. 

For the word "cat", the padded sequence is [".", "c", "a", "t", "."]. The 4-gram contexts and targets become:
* ["c", "a", "t"] → "."

* [".", "c", "a"] → "t"

... 😤 some grunt work...

In [2]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [4]:
# Build stoi and itos

chars = sorted(list(set(''.join(words))))
# s (str) to i (index)
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
# rever the array
itos = {i:s for s,i in stoi.items()}

print(f"stoi['a'] = {stoi['a']}")
print(f"itos[1] = {itos[1]}")

stoi['a'] = 1
itos[1] = a


#### Input
In case of bigram character model, we used the index of character in the vocabulary as input. In case of 4-gram model, we will need to convert the sequence of 3 characters into a single distinct integer.

🧮 *Base-27 representation* Since there are 27 possible characters (including the '.'), each character's index ranges from 0 to 26. Each position represents a power of 27. 
- The first character (context[0]) is the most significant digit, so it's multiplied by 27 squared (27^2). 
- The second character (context[1]) is multiplied by 27^1
- The third character (context[2]) is multiplied by 27^0, which is 1. 
Adding these products together gives a unique integer for every unique combination of the three characters. This ensures every possible 3-character combination gets a **unique integer ID** between 0 and \(27^3 - 1 = 19,682\).

> 💡 For example, if the characters are 'a', 'b', 'c' with indices 1, 2, 3 respectively, the calculation would be $$1 \cdot 27^2 + 2 \cdot 27^1 + 3 \cdot 1 = 729 + 54 + 3 = 786$$ This number will be unique for every different triplet of characters.



In [18]:
# 1. Create the dataset: map 3-character sequences to unique IDs
xs, ys = [], []
for w in ["emma"]:
    chs = ['.'] + list(w) + ['.']
    for i in range(len(chs) - 3):
        print(f'----sample {i+1} ----------------------')
        context = chs[i:i+3]  # 3-character window
        print(f'input sequence: {context}')
        target = chs[i+3]
        print(f'expected output: {target}')
        # Convert context to a unique integer (0 to 27³-1)
        ix_context = stoi[context[0]] * 27**2 + stoi[context[1]] * 27 + stoi[context[2]]
        print(f'unique number for {context} sequence: {ix_context}')
        ix_target = stoi[target]
        xs.append(ix_context)
        ys.append(ix_target)
xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()
print('number of samples in "emma":', num)

----sample 1 ----------------------
input sequence: ['.', 'e', 'm']
expected output: m
unique number for ['.', 'e', 'm'] sequence: 148
----sample 2 ----------------------
input sequence: ['e', 'm', 'm']
expected output: a
unique number for ['e', 'm', 'm'] sequence: 4009
----sample 3 ----------------------
input sequence: ['m', 'm', 'a']
expected output: .
unique number for ['m', 'm', 'a'] sequence: 9829
number of samples in "emma": 3


**One-hot encoding**: In 3-character input sequence, there are possible 27^3 = 19683 unique combinations. So, the input to the model will be a one-hot vector of size 19683. The output will be a one-hot vector of size 27 (for each character in the vocabulary).

```python
xenc = F.one_hot(xs, num_classes=27**3).float()  # Shape [N, 19683]
```

**Weights**: Given the input shape of (N, 19683) and output shape of (N, 27), the weights will be a matrix of size (19683, 27). This means that each unique combination of 3 characters will have a corresponding weight vector for each character in the vocabulary.

```python
W = torch.randn((27**3, 27), generator=g, requires_grad=True)  # Shape [19683, 27]
```


In [ ]:
import time

# 1. Create the dataset: map 3-character sequences to unique IDs
xs, ys = [], []
for w in words:
    chs = ['.'] + list(w) + ['.']
    for i in range(len(chs) - 3):
        context = chs[i:i+3]  # 3-character window
        target = chs[i+3]
        # Convert context to a unique integer (0 to 27³-1)
        ix_context = stoi[context[0]] * 27**2 + stoi[context[1]] * 27 + stoi[context[2]]
        ix_target = stoi[target]
        xs.append(ix_context)
        ys.append(ix_target)
xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()
print('number of samples in extracted from names dataset :', num)

# 2. Initialize weights for 19,683 possible contexts
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27**3, 27), generator=g, requires_grad=True)  # Shape [19683, 27]

def train(n, iteration):
    for k in range(n):
        start_time = time.time()
        # forward pass
        xenc = F.one_hot(xs, num_classes=27**3).float()  # Shape [N, 19683]
        logits = xenc @ W  # Shape [164080, 27]
        counts = logits.exp()
        probs = counts / counts.sum(1, keepdims=True)
        loss = -probs[torch.arange(num), ys].log().mean() + 0.01*(W**2).mean()
        
        # Backward pass and update
        W.grad = None
        loss.backward()

        # Update weights
        W.data += -50 * W.grad

        end_time = time.time()
        print(f'iteration {k+1} took {end_time - start_time:.2f} seconds')
    
    print(f'loss = {loss.item()} after training set {iteration}')

# Too slow
train(10, 1)
# train(100, 2)
# train(100, 3)
# train(100, 4)
# train(100, 5)

number of samples in extracted from names dataset : 164080
iteration 1 took 9.53 seconds
iteration 2 took 10.77 seconds
iteration 3 took 10.80 seconds
iteration 4 took 10.78 seconds
iteration 5 took 10.41 seconds
iteration 6 took 10.75 seconds
iteration 7 took 10.91 seconds
iteration 8 took 10.52 seconds
iteration 9 took 10.56 seconds
iteration 10 took 10.86 seconds
loss = 3.596501350402832 after iteration 1


**Generate Names**

In [26]:
for i in range(10):
  
  out = []
  samples = 0
  while True:
    xenc = F.one_hot(torch.tensor([samples]), num_classes=27**3).float()
    logits = xenc @ W # predict log-counts
    counts = logits.exp() # counts, equivalent to N
    p = counts / counts.sum(1, keepdims=True) # probabilities for next character
    # ----------
    
    samples = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
    out.append(itos[samples])
    if samples == 0:
      break
  print(''.join(out))

ozaltiygkbvmfhsy.
pbogmup.
rajqxedbvbuzwdbterdkriirejkmc.
pxjszwtognw.
phwmklecvogwptjw.
oqr.
trzikluuiqtgkpjluipalrh.
zsvwiisxlppohufpblvkhwdwmywpajwkkgkfuyu.
gdctktprqoafyypcccnabmunvloc.
fpjqbwrfbmalpbcjqxjmxc.


## The Curse of Dimensionality [🚧 WIP]
A goal of statistical language modeling is to learn the joint probability function of sequences of
_[or characters or tokens]_ in a language. This is difficult because a word sequence on which the model is trained may be very different from the word sequence on which the model is tested. This is called the _curse of dimensionality_. The curse of dimensionality refers to the fact that as the number of features (or dimensions) increases, the amount of data needed to estimate the joint probability function grows exponentially. This makes it difficult to learn a good model from limited data.

---
The weight matrix W becomes [19,683, 27], which is massive and sparse.

Most of the 19,683 contexts will never appear in training data.

The model cannot generalize to unseen sequences (e.g., "xyz").



## 🚧 TODO 🐫🔍 Study and implement the paper from Bengio

The paper by Bengio et al. (2003), "[*A Neural Probabilistic Language Model*](https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf)"

---

### **Curse of Dimensionality in Language Modeling**  
Language models predict the probability of the next word given previous words. For example:  
> *"The cat sat on the ___."*  
A model must predict "mat," "couch," etc.  

#### **The Problem**:  
- **Vocabulary size**: Suppose we have a vocabulary of 50,000 words.  
- **Sequence length**: If we model 10-word sequences, the number of possible combinations is \(50,000^{10}\)—a **high-dimensional discrete space**.  
- **Data sparsity**: Even with massive text corpora, most sequences will **never appear** in training data. For example, "*The purple refrigerator sang opera gracefully*" might never occur, making probability estimation impossible.  

This is the curse of dimensionality: the **exponential growth** of possible combinations in high-dimensional spaces makes learning statistically infeasible.

---

### **Example from the Paper**:  

1. **Learning distributed representations**:  
   - Instead of treating words as discrete "one-hot" vectors (e.g., 50,000 dimensions), words are mapped to **lower-dimensional embeddings** (e.g., 100-D).  
   - *Example*: Words like "cat" and "dog" have similar embeddings, capturing semantic relationships.  

2. **Sharing statistical strength**:  
   - Similar words share parameters in the embedding space. For example, if "*cat*" and "*kitten*" have similar embeddings, the model can generalize from "*The cat sat*" to "*The kitten sat*" without needing to see both phrases.  

3. **Avoiding exponential reliance on history**:  
   - Traditional n-gram models suffer because they explicitly model \(P(\text{word}_t | \text{word}_{t-1}, \dots, \text{word}_{t-n})\). For \(n=5\), this requires estimating probabilities for \(50,000^5\) combinations.  
   - Neural networks instead **compress** this information into dense vectors and nonlinear transformations, reducing dimensionality.

